# Text Preprocessing Ulasan Aplikasi Gojek

Notebook ini berisi tahapan **Text Preprocessing (Pemrosesan Teks)** untuk menyiapkan teks ulasan yang masih kotor/tidak terstruktur menjadi teks bersih yang siap diolah oleh algoritma *Machine Learning*.

### 6 Tahapan Text Preprocessing yang Dilakukan:
1. **Cleaning**: Menghapus URL, mention/hashtag, angka, tanda baca, emoji/karakter khusus, dan spasi berlebih.
2. **Case Folding**: Mengubah semua huruf menjadi huruf kecil (*lowercase*).
3. **Tokenization**: Memecah kalimat menjadi daftar potongan kata (*token*).
4. **Normalization**: Mengubah kata gaul/singkatan/tidak baku (*slang*) menjadi kata baku Bahasa Indonesia.
5. **Stopword Removal**: Menghapus kata hubung/kata umum yang tidak memiliki bobot sentimen (namun tetap mempertahankan kata negasi seperti *"tidak"*, *"kurang"*, *"jangan"*).
6. **Stemming**: Mengubah kata berimbuhan menjadi kata dasar menggunakan library **PySastrawi**.

## 0. Import Library & Membaca Data Hasil Pelabelan

Sebelum memulai, pastikan Anda sudah menginstal library tambahan untuk pemrosesan teks Bahasa Indonesia:
```bash
pip install Sastrawi nltk tqdm
```

In [4]:
import re
import string
import pandas as pd
from tqdm import tqdm

# Library khusus Bahasa Indonesia untuk Stopword & Stemming
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Mengaktifkan progress bar pada fungsi .progress_apply() di Pandas
tqdm.pandas()

# Membaca dataset yang sudah dibersihkan dan diberi label pada tahap sebelumnya
file_input = "../data/data_step3_final_labeled.csv"
df = pd.read_csv(file_input)

kolom_teks = 'content'
print(f"Jumlah data yang siap diproses: {len(df)} baris")
df[[kolom_teks, 'score', 'sentiment']].head()

Jumlah data yang siap diproses: 10812 baris


,content,score,sentiment
0,makasii,5,positif
1,mudah digubakan dan bermanfaat,5,positif
2,aplikasi tidak mendukung yg os 13&12,2,negatif
3,sata ingin memesan gogood tapi malah ga ada op...,1,negatif
4,"aplikasi gajelas, kebanyakan di cancel driver,...",1,negatif


---
## 1. Cleaning (Pembersihan Teks)
**Tujuan:** Menghilangkan *noise* (gangguan) pada teks yang tidak berguna untuk analisis sentimen, seperti tautan web (URL), mention, angka, tanda baca, simbol/emoji, dan spasi ganda.

**Penjelasan Pola Regex (`re.sub`) yang Digunakan:**
- `r'https?://\S+|www\.\S+'` : Mencari teks yang diawali `http://`, `https://`, atau `www.` hingga bertemu spasi.
- `r'@[A-Za-z0-9_]+|#[A-Za-z0-9_]+'` : Menghapus mention (`@user`) dan hashtag (`#topik`).
- `r'\d+'` : Menghapus semua angka (`0-9`).
- `r'[^a-zA-Z\s]'` : Menghapus semua karakter **selain** huruf alfabet (`a-z`, `A-Z`) dan spasi (`\s`). Ini otomatis menghapus tanda baca (`!?,.:;"'`), simbol, dan emoji.
- `r'\s+'` : Mengubah kumpulan spasi berlebih, tab, atau baris baru (*enter*) menjadi 1 spasi tunggal.

In [5]:
def cleaning_text(teks):
    # Pastikan input berupa string
    teks = str(teks)
    
    # 1. Hapus URL / Link website (http://, https://, www.)
    teks = re.sub(r'https?://\S+|www\.\S+', ' ', teks)
    
    # 2. Hapus mention (@username) dan hashtag (#hashtag)
    teks = re.sub(r'@[A-Za-z0-9_]+|#[A-Za-z0-9_]+', ' ', teks)
    
    # 3. Hapus angka (0-9)
    teks = re.sub(r'\d+', ' ', teks)
    
    # 4. Hapus tanda baca, emoji, dan karakter khusus (hanya sisakan huruf alfabet & spasi)
    # Diganti dengan spasi ' ' agar kata yang dipisah tanda baca (misal: "bagus,mantap") tidak menempel jadi "bagusmantap"
    teks = re.sub(r'[^a-zA-Z\s]', ' ', teks)
    
    # 5. Hapus spasi berlebih dan spasi di awal/akhir kalimat (.strip())
    teks = re.sub(r'\s+', ' ', teks).strip()
    
    return teks

# Terapkan fungsi cleaning ke kolom ulasan
df['step1_cleaning'] = df[kolom_teks].apply(cleaning_text)

# Contoh perbandingan sebelum & sesudah Cleaning
print("Contoh Hasil Tahap 1 (Cleaning):")
for i in range(3):
    print(f"Asli     : {df[kolom_teks].iloc[i]}")
    print(f"Cleaning : {df['step1_cleaning'].iloc[i]}\n")

Contoh Hasil Tahap 1 (Cleaning):
Asli     : makasii
Cleaning : makasii

Asli     : mudah digubakan dan bermanfaat
Cleaning : mudah digubakan dan bermanfaat

Asli     : aplikasi tidak mendukung yg os 13&12
Cleaning : aplikasi tidak mendukung yg os



---
## 2. Case Folding (Penyeragaman Huruf Kecil)
**Tujuan:** Mengubah seluruh huruf kapital menjadi huruf kecil (*lowercase*).

**Kenapa penting?**
Komputer membedakan karakter berdasarkan kode ASCII-nya. Tanpa *case folding*, kata `"Bagus"`, `"BAGUS"`, dan `"bagus"` akan dianggap sebagai 3 kata yang berbeda. Dengan `.lower()`, ketiganya disamakan menjadi `"bagus"`.

In [6]:
def case_folding(teks):
    # Mengubah seluruh karakter huruf menjadi huruf kecil (lowercase)
    return teks.lower()

# Terapkan fungsi case folding
df['step2_case_folding'] = df['step1_cleaning'].apply(case_folding)

# Contoh perbandingan
print("Contoh Hasil Tahap 2 (Case Folding):")
for i in range(3):
    print(f"Cleaning     : {df['step1_cleaning'].iloc[i]}")
    print(f"Case Folding : {df['step2_case_folding'].iloc[i]}\n")

Contoh Hasil Tahap 2 (Case Folding):
Cleaning     : makasii
Case Folding : makasii

Cleaning     : mudah digubakan dan bermanfaat
Case Folding : mudah digubakan dan bermanfaat

Cleaning     : aplikasi tidak mendukung yg os
Case Folding : aplikasi tidak mendukung yg os



---
## 3. Tokenization (Pemecahan Kalimat Menjadi Kata)
**Tujuan:** Memecah kalimat utuh (string) menjadi daftar kata-kata tunggal (*list of tokens*).

**Kenapa penting?**
Tahap selanjutnya (*Normalization*, *Stopword Removal*, dan *Stemming*) bekerja dengan cara memeriksa **kata demi kata**. Oleh karena itu, kalimat seperti `"aplikasi sangat membantu"` perlu dipecah terlebih dahulu menjadi `["aplikasi", "sangat", "membantu"]`.

In [7]:
def tokenize_text(teks):
    # Memecah kalimat berdasarkan spasi menjadi list kata (token)
    return teks.split()

# Terapkan fungsi tokenisasi
df['step3_tokens'] = df['step2_case_folding'].apply(tokenize_text)

# Contoh perbandingan
print("Contoh Hasil Tahap 3 (Tokenization):")
for i in range(3):
    print(f"Case Folding : {df['step2_case_folding'].iloc[i]}")
    print(f"Tokenization : {df['step3_tokens'].iloc[i]}\n")

Contoh Hasil Tahap 3 (Tokenization):
Case Folding : makasii
Tokenization : ['makasii']

Case Folding : mudah digubakan dan bermanfaat
Tokenization : ['mudah', 'digubakan', 'dan', 'bermanfaat']

Case Folding : aplikasi tidak mendukung yg os
Tokenization : ['aplikasi', 'tidak', 'mendukung', 'yg', 'os']



---
## 4. Normalization (Normalisasi Kata Tidak Baku / Slang)
**Tujuan:** Mengubah kata singkatan, kata gaul (*slang*), atau salah ketik (*typo*) yang sering dipakai pengguna di Play Store menjadi kata baku Bahasa Indonesia.

**Kenapa harus dilakukan sebelum Stopword Removal & Stemming?**
- Library *Stopword* dan *Stemmer* (Sastrawi) hanya mengenali **kata baku Bahasa Indonesia**.
- Jika kata `"gk"` atau `"ga"` tidak diubah menjadi `"tidak"` terlebih dahulu, atau kata `"bgt"` tidak diubah menjadi `"banget"`/`"sangat"`, maka tahap selanjutnya tidak akan bisa mengenali kata tersebut dengan benar.

In [17]:
# Kamus normalisasi kata tidak baku / singkatan yang sering muncul di ulasan aplikasi
# Anda dapat menambahkan kata lain ke dalam dictionary ini sesuai temuan pada dataset Anda
kamus_normalisasi = {
    # Kata negasi
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "enggak": "tidak",
    "nggak": "tidak", "ngga": "tidak", "tdk": "tidak", "tak": "tidak",
    "kagak": "tidak", "g": "tidak", "blm": "belum", "jgn": "jangan",
    "bkn": "bukan", "krg": "kurang",
    
    # Kata hubung & kata ganti umum
    "yg": "yang", "dgn": "dengan", "dg": "dengan", "utk": "untuk",
    "buat": "untuk", "krn": "karena", "karna": "karena", "kalo": "kalau",
    "kl": "kalau", "klo": "kalau", "tp": "tetapi", "tapi": "tetapi",
    "dr": "dari", "dri": "dari", "pd": "pada", "pda": "pada",
    "sy": "saya", "aku": "saya", "gw": "saya", "gue": "saya",
    "udah": "sudah", "udh": "sudah", "sdh": "sudah", "dah": "sudah",
    "lg": "lagi", "lgi": "lagi", "aja": "saja", "aj": "saja",
    "bgt": "sangat", "banget": "sangat", "bngt": "sangat", "amat": "sangat",
    "bs": "bisa", "bsa": "bisa", "jd": "jadi", "jdi": "jadi",
    "dlm": "dalam", "ama": "sama", "sm": "sama", "spt": "seperti",
    "kayak": "seperti", "kyk": "seperti", "gini": "begini", "gitu": "begitu",
    "trs": "terus", "trus": "terus", "skrg": "sekarang", "skrng": "sekarang",
    "knp": "kenapa", "gimana": "bagaimana", "gmn": "bagaimana",
    "byk": "banyak", "bnyk": "banyak", "msh": "masih", "msih": "masih",
    "org": "orang", "orng": "orang", "dl": "dulu", "dlu": "dulu",
    "digubakan": "digunakan",
    "sata": "saya", "nisa": "bisa", "sanagt": "sangat", "ajjjg": "anjing",
    
    # Istilah khusus ulasan aplikasi / Gojek
    "apk": "aplikasi", "app": "aplikasi", "apps": "aplikasi", "aplikasinya": "aplikasi",
    "updet": "update", "apdet": "update", "eror": "error", "lemot": "lambat",
    "ngelag": "lambat", "lag": "lambat", "lelet": "lambat",
    "mantul": "mantap", "good": "bagus", "best": "terbaik",
    "bad": "buruk", "jelek": "buruk", "parah": "buruk",
    "nyesel": "menyesal", "kecewa": "kecewa", "bintang": "bintang",
    "cs": "layanan", "driver": "pengemudi", "drivernya": "pengemudi",
    "ojol": "pengemudi", "gopay": "gopay", "gofood": "gofood", "goride": "goride",
    "promo": "promosi", "diskon": "potongan", "mahal": "mahal", "murah": "murah",
    "bantu": "membantu", "makasih": "terima kasih", "trims": "terima kasih", "thx": "terima kasih",
    "gogood": "gofood"
}

def normalize_tokens(tokens):
    hasil = []
    for kata in tokens:
        # Jika kata ada di kamus_normalisasi, ganti dengan kata bakunya.
        # Jika tidak ada, gunakan kata aslinya.
        kata_baku = kamus_normalisasi.get(kata, kata)
        
        # Jika hasil normalisasi terdiri dari 2 kata (misal: "terima kasih"), pecah lagi agar tetap berupa token tunggal
        hasil.extend(kata_baku.split())
    return hasil

# Terapkan fungsi normalisasi
df['step4_normalized'] = df['step3_tokens'].apply(normalize_tokens)

# Contoh perbandingan
print("Contoh Hasil Tahap 4 (Normalization):")
for i in range(3):
    print(f"Tokenization  : {df['step3_tokens'].iloc[i]}")
    print(f"Normalization : {df['step4_normalized'].iloc[i]}\n")

Contoh Hasil Tahap 4 (Normalization):
Tokenization  : ['makasii']
Normalization : ['makasii']

Tokenization  : ['mudah', 'digubakan', 'dan', 'bermanfaat']
Normalization : ['mudah', 'digunakan', 'dan', 'bermanfaat']

Tokenization  : ['aplikasi', 'tidak', 'mendukung', 'yg', 'os']
Normalization : ['aplikasi', 'tidak', 'mendukung', 'yang', 'os']



---
## 5. Stopword Removal (Penghapusan Kata Umum)
**Tujuan:** Menghapus kata-kata umum (seperti `"yang"`, `"dan"`, `"di"`, `"ke"`, `"dari"`, `"ini"`, `"itu"`, `"untuk"`) yang sangat sering muncul tetapi tidak memiliki arti sentimen positif ataupun negatif.

### Catatan Penting untuk Belajar (Khusus Analisis Sentimen!):
Daftar *stopword* bawaan (baik Sastrawi maupun NLTK) secara default **ikut menghapus kata negasi** seperti `"tidak"`, `"bukan"`, `"jangan"`, `"belum"`, dan `"kurang"`.
- **Masalah jika dihapus:** Kalimat `["aplikasi", "tidak", "bagus"]` akan berubah menjadi `["aplikasi", "bagus"]` (sentimennya terbalik dari negatif menjadi positif!).
- **Solusi:** Kita mengeluarkan kata-kata negasi dari daftar *stopword* agar kata negasi tetap dipertahankan.

In [18]:
# Mengambil daftar stopword Bahasa Indonesia dari library Sastrawi
stopword_factory = StopWordRemoverFactory()
daftar_stopword = set(stopword_factory.get_stop_words())

# Tambahkan stopword tambahan yang sering muncul tapi tidak bermakna sentimen
stopword_tambahan = {"nya", "sih", "deh", "dong", "kok", "lah", "pun", "kan", "nih", "tuh"}
daftar_stopword.update(stopword_tambahan)

# PENTING: Pertahankan kata negasi agar makna kalimat negatif tidak berubah menjadi positif
kata_negasi = {"tidak", "bukan", "jangan", "belum", "kurang", "tanpa"}
daftar_stopword = daftar_stopword - kata_negasi

def remove_stopwords(tokens):
    # Menyaring kata: hanya simpan kata yang TIDAK ada di dalam daftar_stopword
    return [kata for kata in tokens if kata not in daftar_stopword]

# Terapkan fungsi stopword removal
df['step5_no_stopwords'] = df['step4_normalized'].apply(remove_stopwords)

# Contoh perbandingan
print("Contoh Hasil Tahap 5 (Stopword Removal):")
for i in range(5):
    print(f"Normalization    : {df['step4_normalized'].iloc[i]}")
    print(f"Stopword Removal : {df['step5_no_stopwords'].iloc[i]}\n")

Contoh Hasil Tahap 5 (Stopword Removal):
Normalization    : ['makasii']
Stopword Removal : ['makasii']

Normalization    : ['mudah', 'digunakan', 'dan', 'bermanfaat']
Stopword Removal : ['mudah', 'digunakan', 'bermanfaat']

Normalization    : ['aplikasi', 'tidak', 'mendukung', 'yang', 'os']
Stopword Removal : ['aplikasi', 'tidak', 'mendukung', 'os']

Normalization    : ['saya', 'ingin', 'memesan', 'gofood', 'tetapi', 'malah', 'tidak', 'ada', 'opsi', 'pengantaran', 'hemat', 'jadi', 'yang', 'saya', 'dapet', 'hanya', 'pengantaran', 'cepat', 'dan', 'itu', 'sangat', 'mahal', 'bisa', 'lipat', 'dari', 'harga', 'makannya', 'tolong', 'di', 'perbaiki']
Stopword Removal : ['memesan', 'gofood', 'malah', 'tidak', 'opsi', 'pengantaran', 'hemat', 'jadi', 'dapet', 'pengantaran', 'cepat', 'sangat', 'mahal', 'lipat', 'harga', 'makannya', 'perbaiki']

Normalization    : ['aplikasi', 'gajelas', 'kebanyakan', 'di', 'cancel', 'pengemudi', 'sering', 'dapet', 'pengemudi', 'jauh', 'anjing', 'emang']
Stopword R

---
## 6. Stemming (Pengubahan ke Kata Dasar)
**Tujuan:** Menghilangkan imbuhan (awalan, sisipan, akhiran) pada kata sehingga menjadi bentuk kata dasar (*root word*) menggunakan algoritma **Nazief & Adriani** dari library **PySastrawi**.
- Contoh: `"membantu"` $\rightarrow$ `"bantu"`, `"pelayanan"` $\rightarrow$ `"layan"`, `"mengecewakan"` $\rightarrow$ `"kecewa"`.

### Tips Optimasi Kecepatan (Penting untuk Dataset Besar):
Proses *stemming* Sastrawi cukup berat jika dijalankan langsung pada puluhan ribu baris. Untuk mempercepatnya hingga **10–50x lebih cepat**, kita menggunakan teknik **Kamus Cache (Memoization)**:
1. Kumpulkan semua **kata unik** yang ada di seluruh dataset.
2. Lakukan *stemming* hanya 1 kali untuk setiap kata unik tersebut dan simpan ke dalam dictionary `kamus_stem`.
3. Ganti kata-kata pada dataset dengan mengambil hasilnya langsung dari `kamus_stem`.

In [ ]:
# Membuat objek stemmer Sastrawi
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

# 1. Kumpulkan seluruh kata unik dari kolom step5_no_stopwords
kata_unik = set(kata for tokens in df['step5_no_stopwords'] for kata in tokens)
print(f"Total kata unik yang akan di-stemming: {len(kata_unik)} kata")

# 2. Lakukan stemming pada setiap kata unik (dengan progress bar tqdm)
kamus_stem = {}
for kata in tqdm(kata_unik, desc="Proses Stemming Kata Unik"):
    kamus_stem[kata] = stemmer.stem(kata)

# 3. Fungsi untuk memetakan token ke kata dasarnya menggunakan kamus_stem
def stemming_tokens(tokens):
    return [kamus_stem.get(kata, kata) for kata in tokens]

# Terapkan fungsi stemming ke seluruh baris
df['step6_stemmed_tokens'] = df['step5_no_stopwords'].apply(stemming_tokens)

# Gabungkan kembali list token menjadi 1 kalimat bersih (string) untuk tahap TF-IDF / pemodelan
df['text_clean'] = df['step6_stemmed_tokens'].apply(lambda tokens: ' '.join(tokens))

# Contoh perbandingan
print("\nContoh Hasil Tahap 6 (Stemming & Hasil Akhir):")
for i in range(5):
    print(f"Stopword Removal : {df['step5_no_stopwords'].iloc[i]}")
    print(f"Stemming (Token) : {df['step6_stemmed_tokens'].iloc[i]}")
    print(f"Teks Bersih Akhir: {df['text_clean'].iloc[i]}\n")

Total kata unik yang akan di-stemming: 10986 kata


Proses Stemming Kata Unik:   0%|          | 0/10986 [00:00<?, ?it/s]

Proses Stemming Kata Unik:  64%|██████▍   | 7020/10986 [09:24<17:12,  3.84it/s]  

---
## 7. Pemeriksaan Akhir & Penyimpanan Data
Setelah melalui 6 tahap pemrosesan teks, terkadang ada ulasan yang menjadi kosong (misalnya ulasan yang awalnya hanya berisi angka `"123"`, emoji `"👍👍👍"`, atau satu kata stopword `"mantap"` jika terhapus). Baris yang kosong setelah diproses akan kita hapus agar tidak menyebabkan error saat *training* model.

In [ ]:
# Hapus baris yang 'text_clean'-nya menjadi kosong setelah preprocessing
jumlah_sebelum = len(df)
df = df[df['text_clean'].str.strip() != '']
df = df.dropna(subset=['text_clean'])
jumlah_sesudah = len(df)

print(f"Jumlah baris kosong setelah preprocessing yang dihapus: {jumlah_sebelum - jumlah_sesudah} baris")
print(f"Total data siap pakai untuk pemodelan: {jumlah_sesudah} baris")

# Menampilkan tabel perbandingan 6 tahapan pada 5 data pertama
kolom_perbandingan = [
    kolom_teks,
    'step1_cleaning',
    'step2_case_folding',
    'step3_tokens',
    'step4_normalized',
    'step5_no_stopwords',
    'step6_stemmed_tokens',
    'text_clean',
    'sentiment'
]

# Simpan hasil lengkap ke CSV
file_output = "data/data_step4_preprocessed.csv"
df.to_csv(file_output, index=False)
print(f"-> Data hasil text preprocessing berhasil disimpan sebagai: {file_output}")

df[kolom_perbandingan].head()